In [ ]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score

import xgboost as xgb

# -------------------------
# Config
# -------------------------
train_size_list = [0.1, 0.3, 0.5, 0.7, 0.9, 1.0]
seed_list = list(range(1, 11))

data = pd.read_csv("heart_final.csv")
Y = data["HeartDisease"].to_numpy()
X = data.drop(["HeartDisease"], axis=1)

# -------------------------
# Loop over seeds and train sizes
# -------------------------
results = []

for seed in seed_list:
    np.random.seed(seed)

    # Split once per seed (same structure as your code)
    X_train_all, X_test, Y_train_all, Y_test = train_test_split(
        X, Y, train_size=0.7, random_state=2
    )

    # Fit scaler on train_all, transform both
    ct = ColumnTransformer(
        [('stand_scal', StandardScaler(), ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak'])],
        remainder='passthrough'
    )
    X_train_all = ct.fit_transform(X_train_all)
    X_test = ct.transform(X_test)  # IMPORTANT: transform, not fit_transform

    for train_size in train_size_list:
        # Subsample from the 70% training split
        if train_size == 1.0:
            X_train = X_train_all
            Y_train = Y_train_all
        else:
            train_size_f = int(X_train_all.shape[0] * train_size)
            random_train = np.random.permutation(X_train_all.shape[0])[:train_size_f]
            X_train = X_train_all[random_train, :]
            Y_train = Y_train_all[random_train]

        # XGBoost wants numpy arrays
        Xtr = np.asarray(X_train)
        Xte = np.asarray(X_test)
        ytr = np.asarray(Y_train).astype(int)
        yte = np.asarray(Y_test).astype(int)

        dtrain = xgb.DMatrix(Xtr, label=ytr)
        dtest  = xgb.DMatrix(Xte, label=yte)

        params = {
            "objective": "binary:logistic",
            "eval_metric": ["logloss", "auc"],
            "max_depth": 4,
            "eta": 0.05,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "min_child_weight": 1.0,
            "lambda": 1.0,
            "alpha": 0.0,
            "seed": seed,
        }

        evals = [(dtrain, "train"), (dtest, "test")]

        bst = xgb.train(
            params=params,
            dtrain=dtrain,
            num_boost_round=2000,
            evals=evals,
            early_stopping_rounds=50,
            verbose_eval=False,  # set True / 50 if you want logs
        )

        y_prob = bst.predict(dtest)
        y_pred = (y_prob >= 0.5).astype(int)

        acc = accuracy_score(yte, y_pred)
        auc = roc_auc_score(yte, y_prob)

        results.append({
            "seed": seed,
            "train_size": train_size,
            "n_train": len(ytr),
            "accuracy": acc,
            "roc_auc": auc,
            "best_iteration": int(bst.best_iteration) if bst.best_iteration is not None else None
        })

In [2]:
# -------------------------
# Summary table
# -------------------------
results_df = pd.DataFrame(results)
# print(results_df.sort_values(["train_size", "seed"]).to_string(index=False))

print("\nAverages by train_size:")
print(
    results_df.groupby("train_size")[["accuracy", "roc_auc"]]
    .agg(["mean", "std"])
    .to_string()
)



Averages by train_size:
            accuracy             roc_auc          
                mean       std      mean       std
train_size                                        
0.1         0.807609  0.022624  0.897939  0.013123
0.3         0.843841  0.016063  0.920958  0.007358
0.5         0.856522  0.013252  0.924397  0.004923
0.7         0.856884  0.010836  0.926274  0.004633
0.9         0.848551  0.007206  0.927699  0.003160
1.0         0.854348  0.003742  0.929350  0.001908
